In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torchvision
from torchvision import datasets, models, transforms
from torchvision.models import ResNet18_Weights, AlexNet_Weights, GoogLeNet_Weights # For newer torchvision versions
import numpy as np
import matplotlib.pyplot as plt
import time
import os
from PIL import Image
from google.colab import files # For uploading files in Colab

# 1. DEVICE CONFIGURATION
# Check if GPU is available and set the device accordingly
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. HYPERPARAMETERS
NUM_EPOCHS = 10 # Number of epochs to train for (can be small for an exam)
BATCH_SIZE = 32 # Batch size for training and validation
LEARNING_RATE = 0.001 # Learning rate for the optimizer
MODEL_NAME = "resnet18" # Options: "resnet18", "alexnet", "googlenet" (ensure you import weights if needed)
NUM_CLASSES = 10 # CIFAR-10 has 10 classes

# 3. DATA PREPROCESSING AND LOADING
# Define transformations for the training and validation sets
# Pre-trained models expect input images normalized in a specific way.
# ImageNet mean and std:
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize(256), # Resize to a larger size
        transforms.CenterCrop(224), # Crop to the input size of the model
        transforms.RandomHorizontalFlip(), # Data augmentation
        transforms.RandomRotation(10),    # Data augmentation
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
}

# Load CIFAR-10 dataset
# Note: CIFAR-10 images are 32x32. We are upscaling them for models pre-trained on ImageNet (224x224).
print("\nDownloading and preparing CIFAR-10 dataset...")
train_dataset = datasets.CIFAR10(root='./data', train=True,
                                 download=True, transform=data_transforms['train'])
val_dataset = datasets.CIFAR10(root='./data', train=False,
                               download=True, transform=data_transforms['val'])

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE,
                                           shuffle=True, num_workers=2)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=BATCH_SIZE,
                                         shuffle=False, num_workers=2)

dataloaders = {'train': train_loader, 'val': val_loader}
dataset_sizes = {'train': len(train_dataset), 'val': len(val_dataset)}
class_names = train_dataset.classes
print(f"Classes: {class_names}")
print(f"Training dataset size: {dataset_sizes['train']}")
print(f"Validation dataset size: {dataset_sizes['val']}")

# Function to show some images (optional)
def imshow(inp, title=None):
    """Imshow for Tensor."""
    inp = inp.numpy().transpose((1, 2, 0))
    inp = std * inp + mean # Denormalize
    inp = np.clip(inp, 0, 1)
    plt.imshow(inp)
    if title is not None:
        plt.title(title)
    plt.pause(0.001)  # pause a bit so that plots are updated

# Get a batch of training data
# inputs, classes_idx = next(iter(train_loader))
# Make a grid from batch
# out = torchvision.utils.make_grid(inputs)
# imshow(out, title=[class_names[x] for x in classes_idx])


# 4. MODEL SELECTION AND MODIFICATION
print(f"\nLoading pre-trained model: {MODEL_NAME}")

model_ft = None
input_size = 224 # Default for ResNet, GoogLeNet. AlexNet might be 227, but 224 often works.

if MODEL_NAME == "resnet18":
    model_ft = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
elif MODEL_NAME == "alexnet":
    model_ft = models.alexnet(weights=AlexNet_Weights.IMAGENET1K_V1)
elif MODEL_NAME == "googlenet":
    model_ft = models.googlenet(weights=GoogLeNet_Weights.IMAGENET1K_V1)
else:
    print("Invalid model name, exiting...")
    exit()

# Freeze all the network parameters (by default, for transfer learning)
for param in model_ft.parameters():
    param.requires_grad = False

# Get the number of input features for the classifier
if MODEL_NAME == "resnet18" or MODEL_NAME == "googlenet":
    num_ftrs = model_ft.fc.in_features
    model_ft.fc = nn.Linear(num_ftrs, NUM_CLASSES) # Replace the final fully connected layer
elif MODEL_NAME == "alexnet":
    num_ftrs = model_ft.classifier[6].in_features
    model_ft.classifier[6] = nn.Linear(num_ftrs, NUM_CLASSES)

model_ft = model_ft.to(device) # Move model to the configured device

print(f"Model {MODEL_NAME} loaded and final layer adapted for {NUM_CLASSES} classes.")
# print(model_ft) # You can print the model structure if you want

# 5. LOSS FUNCTION AND OPTIMIZER
criterion = nn.CrossEntropyLoss()

# Observe that only parameters of final layer are being optimized as
# opposed to before.
# For fine-tuning the whole network, pass model_ft.parameters()
# For training only the classifier head:
params_to_update = []
if MODEL_NAME == "resnet18" or MODEL_NAME == "googlenet":
    params_to_update = model_ft.fc.parameters()
elif MODEL_NAME == "alexnet":
    params_to_update = model_ft.classifier[6].parameters()

# Optimizer (Adam or SGD)
optimizer_ft = optim.Adam(params_to_update, lr=LEARNING_RATE)
# optimizer_ft = optim.SGD(params_to_update, lr=LEARNING_RATE, momentum=0.9) # Alternative

# Learning rate scheduler (optional, but can help)
# Decay LR by a factor of 0.1 every 7 epochs
exp_lr_scheduler = lr_scheduler.StepLR(optimizer_ft, step_size=7, gamma=0.1)


# 6. TRAINING AND VALIDATION FUNCTION
def train_model(model, criterion, optimizer, scheduler, num_epochs=NUM_EPOCHS):
    since = time.time()

    best_model_wts = model.state_dict() # For saving the best model
    best_acc = 0.0

    # Store history for plotting
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data.
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # Zero the parameter gradients
                optimizer.zero_grad()

                # Forward
                # Track history if only in train
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    if MODEL_NAME == "googlenet" and phase == 'train': # GoogLeNet specific output
                        outputs = outputs.logits # Get primary output
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # Backward + optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # Statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train':
                scheduler.step() # Step the learning rate scheduler

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # Store history
            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(epoch_acc.item())
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(epoch_acc.item())


            # Deep copy the model if it's the best validation accuracy so far
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = model.state_dict()
                # Save the best model (optional, but good practice)
                torch.save(model.state_dict(), f'{MODEL_NAME}_cifar10_best.pth')
                print(f"Best model saved with accuracy: {best_acc:.4f}")


    time_elapsed = time.time() - since
    print(f'\nTraining complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:4f}')

    # Load best model weights
    model.load_state_dict(best_model_wts)
    return model, history

# 7. START TRAINING
print("\nStarting training...")
model_ft, history = train_model(model_ft, criterion, optimizer_ft, exp_lr_scheduler, num_epochs=NUM_EPOCHS)

# 8. PLOT TRAINING HISTORY
print("\nPlotting training history...")
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.title(f'{MODEL_NAME} - Loss over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Train Accuracy')
plt.plot(history['val_acc'], label='Val Accuracy')
plt.title(f'{MODEL_NAME} - Accuracy over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

# 9. FUNCTION FOR SINGLE IMAGE PREDICTION
def predict_image(model, image_path, class_names_list, device):
    print("\nPredicting single image...")
    model.eval() # Set model to evaluation mode

    # Define transformations for a single image (similar to validation)
    image_transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])

    try:
        image = Image.open(image_path).convert('RGB') # Ensure image is RGB
    except FileNotFoundError:
        print(f"Error: Image not found at {image_path}")
        return

    # Display the image
    plt.imshow(image)
    plt.axis('off')
    plt.title("Uploaded Image")
    plt.show()

    # Preprocess the image
    image_tensor = image_transform(image).unsqueeze(0) # Add batch dimension
    image_tensor = image_tensor.to(device)

    with torch.no_grad(): # No need to calculate gradients for prediction
        outputs = model(image_tensor)
        probabilities = torch.softmax(outputs, dim=1)[0] # Get probabilities for the first (and only) image
        _, predicted_idx = torch.max(outputs, 1)

    predicted_class_name = class_names_list[predicted_idx.item()]
    predicted_probability = probabilities[predicted_idx.item()].item()

    print(f"Predicted Class: {predicted_class_name}")
    print(f"Confidence: {predicted_probability*100:.2f}%")

    # Print top N probabilities (e.g., top 3)
    print("\nTop Probabilities:")
    top_n_probs, top_n_indices = torch.topk(probabilities, 3)
    for i in range(top_n_probs.size(0)):
        class_name = class_names_list[top_n_indices[i].item()]
        prob = top_n_probs[i].item()
        print(f"- {class_name}: {prob*100:.2f}%")

# 10. TEST WITH AN UPLOADED IMAGE
# Upload an image file from your local computer to Colab
print("\nPlease upload an image for testing.")
uploaded = files.upload()

if len(uploaded.keys()) == 0:
    print("No file uploaded. Skipping single image prediction.")
else:
    # Get the name of the uploaded file
    uploaded_image_path = next(iter(uploaded))
    print(f"Uploaded file: {uploaded_image_path}")

    # Make a prediction
    predict_image(model_ft, uploaded_image_path, class_names, device)

print("\n--- End of Script ---")